# A/B Eval — Round 3 vs DeepFool (c_dog defense)

Runs a single A/B eval: **round 3** DPC-UNet checkpoint vs **deepfool** attack (defense=c_dog).

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Upload `dpc_unet_r3.pt` to your **Google Drive root**
   - Round 3 checkpoint (`dpc_unet_adversarial_finetuned.pt`) → save as `dpc_unet_r3.pt`
   - Round 2 checkpoint (`dpc_unet_adversarial_finetuned.pt.prev`) → save as `dpc_unet_r2.pt`

**After running:** download `metrics.json` and `run_summary.json` from the final cell.

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU — change runtime type to T4 GPU.')

In [ ]:
import os
REPO_URL = 'https://github.com/Cmaddock99/YOLO-Bad-Triangle.git'
REPO_DIR = '/content/YOLO-Bad-Triangle'
if not os.path.exists(REPO_DIR):
    os.system(f'git clone {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull')
os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'ultralytics>=8.4.23', 'opencv-python-headless', 'pandas', 'tqdm', 'pyyaml'], check=True)
print('Dependencies installed.')

In [ ]:
import json, urllib.request, zipfile
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

COCO_DIR = Path('coco/val2017_subset500')
IMG_DIR  = COCO_DIR / 'images'
LBL_DIR  = COCO_DIR / 'labels'
IMG_DIR.mkdir(parents=True, exist_ok=True)
LBL_DIR.mkdir(parents=True, exist_ok=True)

ANNO_ZIP  = Path('/tmp/annotations_trainval2017.zip')
ANNO_JSON = Path('/tmp/coco_anno/annotations/instances_val2017.json')
SUBSET_JSON = COCO_DIR / 'instances_val2017_subset500.json'

if not SUBSET_JSON.exists():
    if not ANNO_ZIP.exists():
        print('Downloading COCO annotations (~241 MB)...')
        urllib.request.urlretrieve(
            'http://images.cocodataset.org/annotations/annotations_trainval2017.zip', ANNO_ZIP)
    if not ANNO_JSON.exists():
        with zipfile.ZipFile(ANNO_ZIP) as zf:
            zf.extractall('/tmp/coco_anno')
    with open(ANNO_JSON) as f:
        full = json.load(f)
    subset_images = full['images'][:500]
    subset_ids    = {img['id'] for img in subset_images}
    subset_annos  = [a for a in full['annotations'] if a['image_id'] in subset_ids]
    SUBSET_JSON.write_text(json.dumps({**full, 'images': subset_images, 'annotations': subset_annos}))
    print(f'Subset JSON saved.')

with open(SUBSET_JSON) as f:
    subset = json.load(f)

missing = [img for img in subset['images'] if not (IMG_DIR / img['file_name']).exists()]
print(f'Downloading {len(missing)} images...')
for img in tqdm(missing, unit='img'):
    urllib.request.urlretrieve(
        'http://images.cocodataset.org/val2017/' + img['file_name'],
        IMG_DIR / img['file_name'])

# Write YOLO labels
cat_map = {cat['id']: i for i, cat in enumerate(subset['categories'])}
annos_by_image = defaultdict(list)
for anno in subset['annotations']:
    annos_by_image[anno['image_id']].append(anno)
for img in subset['images']:
    lbl_path = LBL_DIR / (Path(img['file_name']).stem + '.txt')
    if lbl_path.exists():
        continue
    W, H = img['width'], img['height']
    lines = []
    for anno in annos_by_image[img['id']]:
        if anno.get('iscrowd', 0): continue
        x, y, w, h = anno['bbox']
        lines.append(f"{cat_map[anno['category_id']]} {(x+w/2)/W:.6f} {(y+h/2)/H:.6f} {w/W:.6f} {h/H:.6f}")
    lbl_path.write_text('\n'.join(lines))
print('COCO data ready.')

In [ ]:
from ultralytics import YOLO
from pathlib import Path
if not Path('yolo26n.pt').exists():
    print('Downloading yolo26n.pt...')
    YOLO('yolo26n.pt')
print('yolo26n.pt ready:', Path('yolo26n.pt').exists())

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

DRIVE_CHECKPOINT = Path('/content/drive/MyDrive/dpc_unet_r3.pt')
LOCAL_CHECKPOINT  = Path('dpc_unet_r3.pt')

if not DRIVE_CHECKPOINT.exists():
    raise FileNotFoundError(f'Upload dpc_unet_r3.pt to your Google Drive root first.')

shutil.copy(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)
print(f'Checkpoint ready: {LOCAL_CHECKPOINT} ({LOCAL_CHECKPOINT.stat().st_size / 1e6:.1f} MB)')

In [ ]:
import subprocess, sys, os

RUN_NAME   = 'ab_r3_deepfool_cdog'
ATTACK     = 'deepfool'
MAX_IMAGES = 100
CHECKPOINT = 'dpc_unet_r3.pt'

env = os.environ.copy()
env['PYTHONPATH'] = 'src'
env['DPC_UNET_CHECKPOINT_PATH'] = CHECKPOINT

cmd = [
    sys.executable, 'scripts/run_unified.py', 'run-one',
    '--config', 'configs/default.yaml',
    '--seed', '137',
    '--set', f'attack.name={ATTACK}',
    '--set', 'defense.name=c_dog',
    '--set', f'runner.max_images={MAX_IMAGES}',
    '--set', 'validation.enabled=true',
    '--set', f'runner.run_name={RUN_NAME}',
    '--set', 'attack.params.epsilon=0.1',
    '--set', 'attack.params.steps=50',
]

print('Running:', ' '.join(cmd))
result = subprocess.run(cmd, env=env)
print('Exit code:', result.returncode)

In [ ]:
import json
from pathlib import Path

run_dir = Path('outputs/framework_runs/ab_r3_deepfool_cdog')
metrics = json.loads((run_dir / 'metrics.json').read_text())
summary = json.loads((run_dir / 'run_summary.json').read_text())

map50 = metrics.get('mAP50') or metrics.get('validation', {}).get('mAP50', 'N/A')
fp = summary.get('provenance', {}).get('checkpoint_fingerprint_sha256', 'N/A')

print(f'Run: ab_r3_deepfool_cdog')
print(f'Checkpoint: round 3 (dpc_unet_r3.pt)')
print(f'Attack: deepfool | Defense: c_dog | Images: 100 | Seed: 137')
print(f'mAP50: {map50:.4f}')
print(f'Checkpoint fingerprint: {fp[:24]}...')

In [ ]:
from google.colab import files
from pathlib import Path

run_dir = Path('outputs/framework_runs/ab_r3_deepfool_cdog')
files.download(str(run_dir / 'metrics.json'))
files.download(str(run_dir / 'run_summary.json'))
files.download(str(run_dir / 'predictions.jsonl'))
print('Downloads triggered.')